# Notebook 1: Data Pipeline & Feature Engineering
Merges all Home Credit files. Builds composite features, signal strength score, and dual thin-file mode classification.

In [8]:
import pandas as pd
import numpy as np
import warnings, json, pickle
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'

app    = pd.read_csv(DATA_DIR + 'application_train.csv')
bureau = pd.read_csv(DATA_DIR + 'bureau.csv')
prev   = pd.read_csv(DATA_DIR + 'previous_application.csv')
inst   = pd.read_csv(DATA_DIR + 'installments_payments.csv')
cc     = pd.read_csv(DATA_DIR + 'credit_card_balance.csv')

print('app:', app.shape, '| bureau:', bureau.shape,
      '| prev:', prev.shape, '| inst:', inst.shape, '| cc:', cc.shape)

app: (307511, 122) | bureau: (1716428, 17) | prev: (1670214, 37) | inst: (13605401, 8) | cc: (3840312, 23)


## 1. Aggregations from History Files

In [9]:
# ── Bureau ─────────────────────────────────────────────────────────────────
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_count         = ('SK_ID_BUREAU',       'count'),
    bureau_active_count  = ('CREDIT_ACTIVE',       lambda x: (x == 'Active').sum()),
    bureau_avg_overdue   = ('CREDIT_DAY_OVERDUE',  'mean'),
    bureau_max_overdue   = ('CREDIT_DAY_OVERDUE',  'max'),
    bureau_total_debt    = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_avg_credit    = ('AMT_CREDIT_SUM',      'mean'),
    bureau_prolong_count = ('CNT_CREDIT_PROLONG',  'sum'),
).reset_index()
bureau_agg['bureau_overdue_ratio'] = (
    bureau_agg['bureau_avg_overdue'] / (bureau_agg['bureau_avg_credit'] + 1)
)

# ── Previous Applications ──────────────────────────────────────────────────
prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_app_count      = ('SK_ID_PREV',          'count'),
    prev_approved_count = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    prev_refused_count  = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    prev_avg_credit     = ('AMT_CREDIT',           'mean'),
    prev_avg_annuity    = ('AMT_ANNUITY',          'mean'),
).reset_index()
prev_agg['prev_approval_rate'] = (
    prev_agg['prev_approved_count'] / (prev_agg['prev_app_count'] + 1e-5)
)

# ── Installments ───────────────────────────────────────────────────────────
inst['DAYS_LATE']     = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1e-5)
inst['IS_LATE']       = (inst['DAYS_LATE'] > 0).astype(int)
inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_count         = ('SK_ID_PREV',    'count'),
    inst_avg_days_late = ('DAYS_LATE',     'mean'),
    inst_max_days_late = ('DAYS_LATE',     'max'),
    inst_late_rate     = ('IS_LATE',       'mean'),
    inst_avg_pay_ratio = ('PAYMENT_RATIO', 'mean'),
    inst_min_pay_ratio = ('PAYMENT_RATIO', 'min'),
).reset_index()

# ── Credit Card ────────────────────────────────────────────────────────────
cc['UTILIZATION'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1e-5)
cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_months_active   = ('MONTHS_BALANCE', 'count'),
    cc_avg_balance     = ('AMT_BALANCE',    'mean'),
    cc_avg_utilization = ('UTILIZATION',    'mean'),
    cc_max_utilization = ('UTILIZATION',    'max'),
).reset_index()

print('All aggregations complete.')

All aggregations complete.


## 2. Base Feature Engineering

In [10]:
BASE_COLS = [
    'SK_ID_CURR', 'TARGET',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_HOUSING_TYPE',
    'DAYS_EMPLOYED', 'DAYS_BIRTH',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT',
]

df = app[BASE_COLS].copy()
df['FLAG_OWN_CAR']    = (df['FLAG_OWN_CAR']    == 'Y').astype(int)
df['FLAG_OWN_REALTY'] = (df['FLAG_OWN_REALTY'] == 'Y').astype(int)

df['AGE_YEARS']         = -df['DAYS_BIRTH'] / 365
df['EMPLOYED_YEARS']    = df['DAYS_EMPLOYED'].apply(lambda x: -x/365 if x < 0 else 0)
df['DEBT_TO_INCOME']    = df['AMT_CREDIT']  / (df['AMT_INCOME_TOTAL'] + 1)
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
df['CREDIT_TO_GOODS']   = df['AMT_CREDIT']  / (df['AMT_GOODS_PRICE']  + 1)
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)
df['CHILDREN_RATIO']    = df['CNT_CHILDREN'] / (df['CNT_FAM_MEMBERS'] + 1)
df['EMI_BURDEN']        = df['AMT_ANNUITY']  / (df['AMT_INCOME_TOTAL'] + 1)
df['INCOME_STABILITY']  = df['EMPLOYED_YEARS'] / (df['AGE_YEARS'] + 1)
df['LOAN_TO_INCOME']    = df['AMT_CREDIT']   / (df['AMT_INCOME_TOTAL'] * 12 + 1)
df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'], inplace=True)
print('Base features:', df.shape)

Base features: (307511, 28)


## 3. Composite Features
Compress correlated signals into interpretable single scores. Weights are domain-informed, not arbitrary.

In [11]:
# ── FINANCIAL_PRESSURE: 0=no stress, 1=maximum stress ─────────────────────
df['FINANCIAL_PRESSURE'] = (
    0.40 * df['EMI_BURDEN'].clip(0, 1)     +
    0.35 * df['LOAN_TO_INCOME'].clip(0, 1) +
    0.25 * df['CHILDREN_RATIO'].clip(0, 1)
)

# ── STABILITY_SCORE: employment duration + asset ownership ────────────────
df['STABILITY_SCORE'] = (
    0.50 * df['INCOME_STABILITY'].clip(0, 1)        +
    0.30 * (df['EMPLOYED_YEARS'] / 40).clip(0, 1)  +
    0.20 * df['FLAG_OWN_REALTY'].astype(float)
)

# ── ASSET_SCORE: tangible collateral proxy ─────────────────────────────────
df['ASSET_SCORE'] = (
    0.60 * df['FLAG_OWN_REALTY'].astype(float) +
    0.40 * df['FLAG_OWN_CAR'].astype(float)
)

# ── INCOME_ADEQUACY: income buffer above loan obligations ─────────────────
df['INCOME_ADEQUACY'] = (
    df['AMT_INCOME_TOTAL'] / (df['AMT_ANNUITY'] * 12 + 1)
).clip(0, 10)

print('Composite features:')
print(df[['FINANCIAL_PRESSURE','STABILITY_SCORE','ASSET_SCORE','INCOME_ADEQUACY']].describe().round(3))

Composite features:
       FINANCIAL_PRESSURE  STABILITY_SCORE  ASSET_SCORE  INCOME_ADEQUACY
count          307497.000       307511.000   307511.000       307499.000
mean                0.210            0.242        0.552            0.611
std                 0.111            0.144        0.335            0.407
min                 0.005            0.000        0.000            0.044
25%                 0.129            0.185        0.400            0.364
50%                 0.191            0.230        0.600            0.512
75%                 0.270            0.314        0.600            0.726
max                 0.828            0.859        1.000           10.000


## 4. Merge + Thin-File Mode Classification

In [12]:
thick = (
    df
    .merge(bureau_agg, on='SK_ID_CURR', how='left')
    .merge(prev_agg,   on='SK_ID_CURR', how='left')
    .merge(inst_agg,   on='SK_ID_CURR', how='left')
    .merge(cc_agg,     on='SK_ID_CURR', how='left')
)

# ── True thin-file: no records in ANY history source ──────────────────────
thick['IS_THIN_FILE'] = (
    thick['bureau_count'].isna() &
    thick['inst_count'].isna()   &
    thick['prev_app_count'].isna()
).astype(int)

# ── Signal strength: how much behavioral data exists (0=none, 1=full) ─────
thick['SIGNAL_STRENGTH'] = (
    (~thick['bureau_count'].isna()).astype(float)     * 0.35 +
    (~thick['inst_count'].isna()).astype(float)       * 0.35 +
    (~thick['prev_app_count'].isna()).astype(float)   * 0.20 +
    (~thick['cc_months_active'].isna()).astype(float) * 0.10
)

# ── Two thin-file modes: near-thin has partial signals, pure-thin has none ─
def assign_mode(row):
    if row['IS_THIN_FILE'] == 0:        return 'C_thick'
    elif row['SIGNAL_STRENGTH'] > 0:    return 'A_near_thin'
    else:                               return 'B_pure_thin'

thick['USER_MODE'] = thick.apply(assign_mode, axis=1)

print('Mode distribution:')
print(thick['USER_MODE'].value_counts())
print(f'\nThin-file total: {thick["IS_THIN_FILE"].sum()} ({thick["IS_THIN_FILE"].mean()*100:.1f}%)')

Mode distribution:
USER_MODE
C_thick        305271
B_pure_thin      2235
A_near_thin         5
Name: count, dtype: int64

Thin-file total: 2240 (0.7%)


## 5. Encode + Save

In [13]:
from sklearn.preprocessing import LabelEncoder

CAT_COLS = [
    'NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS','NAME_INCOME_TYPE',
    'OCCUPATION_TYPE','ORGANIZATION_TYPE','NAME_HOUSING_TYPE'
]
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    thick[col] = thick[col].fillna('Unknown')
    thick[col] = le.fit_transform(thick[col])
    encoders[col] = le

numeric_cols = thick.select_dtypes(include=[np.number]).columns
thick[numeric_cols] = thick[numeric_cols].fillna(0)

thick.to_parquet('thick_file_dataset.parquet', index=False)
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print('Saved: thick_file_dataset.parquet')
print('Class balance:', thick['TARGET'].value_counts(normalize=True).round(3).to_dict())

Saved: thick_file_dataset.parquet
Class balance: {0: 0.919, 1: 0.081}


## 6. Feature Set Definitions

In [14]:
HISTORY_COLS = [
    'bureau_count','bureau_active_count','bureau_avg_overdue','bureau_max_overdue',
    'bureau_total_debt','bureau_avg_credit','bureau_prolong_count','bureau_overdue_ratio',
    'prev_app_count','prev_approved_count','prev_refused_count',
    'prev_avg_credit','prev_avg_annuity','prev_approval_rate',
    'inst_count','inst_avg_days_late','inst_max_days_late','inst_late_rate',
    'inst_avg_pay_ratio','inst_min_pay_ratio',
    'cc_months_active','cc_avg_balance','cc_avg_utilization','cc_max_utilization',
]

EXCLUDE = {'SK_ID_CURR','TARGET','IS_THIN_FILE','USER_MODE','SIGNAL_STRENGTH'}
TEACHER_FEATURES = [c for c in thick.columns if c not in EXCLUDE]
STUDENT_FEATURES = [c for c in TEACHER_FEATURES if c not in HISTORY_COLS]

with open('feature_sets.json', 'w') as f:
    json.dump({
        'teacher': TEACHER_FEATURES,
        'student': STUDENT_FEATURES,
        'history_cols': HISTORY_COLS
    }, f)

print(f'Teacher features : {len(TEACHER_FEATURES)}')
print(f'Student features : {len(STUDENT_FEATURES)}')
print(f'\nStudent feature list:\n{STUDENT_FEATURES}')

Teacher features : 54
Student features : 30

Student feature list:
['CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_HOUSING_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT', 'AGE_YEARS', 'EMPLOYED_YEARS', 'DEBT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_GOODS', 'INCOME_PER_PERSON', 'CHILDREN_RATIO', 'EMI_BURDEN', 'INCOME_STABILITY', 'LOAN_TO_INCOME', 'FINANCIAL_PRESSURE', 'STABILITY_SCORE', 'ASSET_SCORE', 'INCOME_ADEQUACY']
